# Fashion RAG Chatbot — Phiên bản hoàn chỉnh (Text + Multimodal)
> Chạy **từ trên xuống dưới**. Phần 2 (Data Pipeline) chỉ cần chạy 1 lần khi lần đầu setup.

## PHẦN 1: Import thư viện

In [1]:
import json, sys, uuid, os, time, base64
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_core.embeddings import Embeddings
from langchain_qdrant import QdrantVectorStore
from qdrant_client import QdrantClient
from qdrant_client.http.models import Distance, VectorParams
from qdrant_client.models import PointStruct
from qdrant_client.http.models import Filter, FieldCondition, MatchAny
from langchain_core.documents import Document
from tqdm import tqdm
from langchain_ollama import ChatOllama, OllamaEmbeddings
from langchain_classic.chains import create_retrieval_chain, create_history_aware_retriever
from langchain_classic.chains.combine_documents import create_stuff_documents_chain
from langchain_community.chat_message_histories import RedisChatMessageHistory
from langchain_core.runnables.history import RunnableWithMessageHistory
from langchain_core.prompts import PromptTemplate, ChatPromptTemplate, MessagesPlaceholder
from langchain_core.callbacks import BaseCallbackHandler
import ollama
from ollama import Client
from PIL import Image
import io
print("[OK] Import hoàn tất!")

[OK] Import hoàn tất!


## PHẦN 2: Mô hình Embedding (BGE-M3)

In [2]:
class BGEM3Embeddings(Embeddings):
    """
    Wrapper cho BGE-M3 — mô hình embedding đa ngôn ngữ (vector 1024 chiều).
    Hỗ trợ tiếng Việt, tiếng Anh và nhiều ngôn ngữ khác mà không cần tách từ thủ công.
    """
    def __init__(self, model_name="bge-m3"):
        self.ollama_embeddings = OllamaEmbeddings(
            model=model_name,
            base_url="http://localhost:11434"
        )

    def embed_documents(self, texts: list[str]) -> list[list[float]]:
        return self.ollama_embeddings.embed_documents(texts)

    def embed_query(self, text: str) -> list[float]:
        return self.ollama_embeddings.embed_query(text)

print("[OK] BGEM3Embeddings đã định nghĩa xong!")

[OK] BGEM3Embeddings đã định nghĩa xong!


## PHẦN 3: Data Pipeline — Indexing sản phẩm vào Qdrant
> ⚠️ **Chỉ chạy 1 lần duy nhất** khi lần đầu setup hoặc khi cần cập nhật dữ liệu.  
> Nếu Qdrant đã có dữ liệu, bỏ qua phần này và chạy thẳng Phần 4.

In [ ]:
def process_fashion_metadata(file_path: str) -> list:
    """
    Đọc file JSONL và chuyển từng sản phẩm thành LangChain Document.
    page_content: văn bản mô tả sản phẩm (để embedding hiểu ngữ nghĩa)
    metadata:     dict chứa product_id, category, price... (để filter và hiển thị)
    """
    documents = []
    with open(file_path, "r", encoding="utf-8") as f:
        for line in f:
            if not line.strip():
                continue
            item = json.loads(line)

            image_urls = [img.get("large") for img in item.get("images", []) if "large" in img]

            metadata = {
                "product_id": item.get("product_id", ""),
                "category":   item.get("category", ""),
                "department": item.get("department", ""),
                "brand":      item.get("brand", ""),
                "price":      item.get("price", 0),
                "images":     image_urls,
            }

            details    = item.get("details", {})
            page_content = (
                f"Sản phẩm {item.get('title', 'Không có tên')} "
                f"thuộc danh mục {item.get('category', '')} "
                f"dành cho {item.get('department', '')}. "
                f"Thương hiệu: {item.get('brand', 'Không rõ')}. "
                f"Mức giá: {item.get('price', 0)} VNĐ. "
                f"Đặc điểm: màu sắc: {details.get('main_color', 'Không rõ')}, "
                f"chất liệu: {details.get('material', 'Không rõ')}, "
                f"kích cỡ: {details.get('size', 'Không rõ')}, "
                f"họa tiết: {details.get('pattern', 'Không rõ')}. "
                f"Phù hợp cho {item.get('season', '')} và dịp {item.get('occasion', '')}. "
                f"Mô tả: {item.get('description', '')}"
            )

            documents.append(Document(page_content=page_content, metadata=metadata))
    return documents


def process_all_directory(directory_path: str) -> list:
    """Duyệt toàn bộ file .jsonl trong thư mục và gộp thành 1 danh sách Document."""
    all_docs = []
    for filename in sorted(os.listdir(directory_path)):
        if filename.endswith(".jsonl"):
            file_path = os.path.join(directory_path, filename)
            print(f"  → Đang đọc: {filename}")
            all_docs.extend(process_fashion_metadata(file_path))
    return all_docs


def run_data_pipeline(folder_path: str = "./Fashion_Metadata",
                      qdrant_url: str   = "http://localhost:6333",
                      collection_name: str = "fashion_products_bge_m3"):
    """
    Hàm tổng: đọc toàn bộ JSONL → embed bằng BGE-M3 → đẩy vào Qdrant.
    Có tính năng resume: tự bỏ qua phần đã index, tiếp tục từ chỗ còn lại.
    """
    if not os.path.exists(folder_path):
        print(f"[LỖI] Không tìm thấy thư mục: {folder_path}")
        return

    print(f"[THÔNG BÁO] Đọc dữ liệu từ: {folder_path}")
    all_docs = process_all_directory(folder_path)
    print(f"[OK] Tổng số Documents: {len(all_docs)}")

    emb    = BGEM3Embeddings()
    client = QdrantClient(url=qdrant_url)

    if not client.collection_exists(collection_name):
        print(f"[THÔNG BÁO] Tạo mới collection '{collection_name}'...")
        client.create_collection(
            collection_name=collection_name,
            vectors_config=VectorParams(size=1024, distance=Distance.COSINE),
        )
        current_count = 0
    else:
        current_count = client.get_collection(collection_name).points_count
        print(f"[THÔNG BÁO] Đã có {current_count} sản phẩm trong DB.")

    remaining = all_docs[current_count:]
    if not remaining:
        print("[OK] Dữ liệu đã được index đầy đủ trước đó. Không cần chạy lại!")
        return

    print(f"[THÔNG BÁO] Bắt đầu index {len(remaining)} sản phẩm còn lại...")
    vdb = QdrantVectorStore(client=client, collection_name=collection_name, embedding=emb)

    batch_size = 128
    with tqdm(total=len(all_docs), initial=current_count,
              desc="Tiến độ index", unit="SP") as pbar:
        for i in range(0, len(remaining), batch_size):
            batch = remaining[i : i + batch_size]
            vdb.add_documents(documents=batch)
            pbar.update(len(batch))

    print("[OK] Index hoàn tất vào Qdrant!")

# ── Bỏ comment dòng dưới nếu muốn chạy data pipeline ──
# run_data_pipeline()

[THÔNG BÁO] Đọc dữ liệu từ: ./Fashion_Metadata
  → Đang đọc: Fashion_Metadata_Ao.jsonl
  → Đang đọc: Fashion_Metadata_Ao_khoac.jsonl
  → Đang đọc: Fashion_Metadata_Bong_tai.jsonl
  → Đang đọc: Fashion_Metadata_Chan_vay.jsonl
  → Đang đọc: Fashion_Metadata_Dam.jsonl
  → Đang đọc: Fashion_Metadata_Day_chuyen.jsonl
  → Đang đọc: Fashion_Metadata_Do_lot.jsonl
  → Đang đọc: Fashion_Metadata_Dong_ho.jsonl
  → Đang đọc: Fashion_Metadata_Gang_tay.jsonl
  → Đang đọc: Fashion_Metadata_Ghim_cai_ao.jsonl
  → Đang đọc: Fashion_Metadata_Giay.jsonl
  → Đang đọc: Fashion_Metadata_Jumpsuit.jsonl
  → Đang đọc: Fashion_Metadata_Khac.jsonl
  → Đang đọc: Fashion_Metadata_Kinh_mat.jsonl
  → Đang đọc: Fashion_Metadata_Mu.jsonl
  → Đang đọc: Fashion_Metadata_Nhan.jsonl
  → Đang đọc: Fashion_Metadata_Phu_kien_ho_tro.jsonl
  → Đang đọc: Fashion_Metadata_Quan.jsonl
  → Đang đọc: Fashion_Metadata_Tui_xach.jsonl
  → Đang đọc: Fashion_Metadata_Vong_tay.jsonl
[OK] Tổng số Documents: 53979
[THÔNG BÁO] Tạo mới collect

Tiến độ index: 100%|██████████| 53979/53979 [37:16<00:00, 24.14SP/s]

[OK] Index hoàn tất vào Qdrant!


## PHẦN 4: Kết nối Qdrant & khởi tạo Vector Store

In [8]:
print("[THÔNG BÁO] Đang khởi tạo Embedding model...")
custom_embeddings = BGEM3Embeddings()

print("[THÔNG BÁO] Đang kết nối Qdrant...")
# Dùng Docker:  client = QdrantClient(url="http://localhost:6333")
# Dùng local:   client = QdrantClient(path="./qdrant_data")
client = QdrantClient(url="http://localhost:6333")

vector_db = QdrantVectorStore(
    client=client,
    collection_name="fashion_products_bge_m3",
    embedding=custom_embeddings,
)

retriever = vector_db.as_retriever(
    search_type="similarity_score_threshold",
    search_kwargs={"k": 5, "score_threshold": 0.7},
)

print(f"[OK] Qdrant kết nối thành công!")
print(f"[OK] Retriever sẵn sàng (top-5, score ≥ 0.7)")

[THÔNG BÁO] Đang khởi tạo Embedding model...
[THÔNG BÁO] Đang kết nối Qdrant...
[OK] Qdrant kết nối thành công!
[OK] Retriever sẵn sàng (top-5, score ≥ 0.7)


## PHẦN 5: Layer B — Nạp & Index kiến thức phối đồ

In [9]:
def load_layer_b(file_path: str) -> list:
    with open(file_path, "r", encoding="utf-8") as f:
        return json.load(f)

layer_b_female = load_layer_b("Fashion_Stylists/Layer_B_Female_Knowledge.json")
layer_b_male   = load_layer_b("Fashion_Stylists/Layer_B_Male_Knowledge.json")
print(f"[OK] Layer B: {len(layer_b_female)} rules Nữ | {len(layer_b_male)} rules Nam")

[OK] Layer B: 880 rules Nữ | 416 rules Nam


In [10]:
def index_layer_b(data: list, collection_name: str):
    """
    Index toàn bộ Layer B vào Qdrant bằng BGE-M3 (chạy 1 lần).
    Dùng custom_embeddings (global) để tránh phụ thuộc vào vector_db.embeddings.
    """
    existing = [c.name for c in client.get_collections().collections]
    if collection_name in existing:
        count = client.count(collection_name).count
        print(f"[SKIP] {collection_name} đã tồn tại ({count} rules) — bỏ qua index.")
        return

    client.create_collection(
        collection_name=collection_name,
        vectors_config=VectorParams(size=1024, distance=Distance.COSINE),
    )

    points = []
    for i, rule in enumerate(data):
        # Ghép các field có ngữ nghĩa để embed — bge-m3 xử lý tốt đa ngôn ngữ Vi+En
        text   = (f"{rule['rule_key']} {rule['phong_cach']} "
                  f"{rule['boi_canh']} {rule['ly_do_tu_van']}")
        vector = custom_embeddings.embed_query(text)   # dùng custom_embeddings trực tiếp
        points.append(PointStruct(id=i, vector=vector, payload=rule))

    client.upsert(collection_name=collection_name, points=points)
    print(f"[OK] Indexed {len(points)} rules → {collection_name}")

index_layer_b(layer_b_female, "layer_b_female")
index_layer_b(layer_b_male,   "layer_b_male")

[OK] Indexed 880 rules → layer_b_female
[OK] Indexed 416 rules → layer_b_male


In [11]:
def find_matching_rule(user_query: str,
                       gender: str  = "female",
                       profile: dict = None) -> dict | None:
    """
    HÀM 1: CỬA HÀNG TRƯỞNG TÌM 'CÔNG THỨC CHỦ ĐẠO'
    - Mục đích: Tìm 1 bản thiết kế phong cách hợp với yêu cầu và vóc dáng của khách nhất.
    """
    collection   = f"layer_b_{gender}"
    # Dịch yêu cầu của khách ("đồ đi tiệc") sang mảng số (vector) để AI hiểu
    query_vector = custom_embeddings.embed_query(user_query)

    conditions = []
    # Nếu khách có khai báo Dáng người và Tone da -> Viết vào sổ tay để làm bộ lọc cứng
    if profile:
        if profile.get("dang_nguoi"):
            conditions.append(FieldCondition(
                key="dang_nguoi",
                match=MatchAny(any=[profile["dang_nguoi"], "Mọi vóc dáng"])
            ))
        if profile.get("tone_da"):
            conditions.append(FieldCondition(
                key="tone_da",
                match=MatchAny(any=[profile["tone_da"], "Mọi tone da"])
            ))

    # BƯỚC 1: Lục tìm trong kho Qdrant (Lọc khắt khe cả Dáng + Da)
    search_filter = Filter(must=conditions) if conditions else None
    response = client.query_points(
        collection_name=collection, query=query_vector,
        query_filter=search_filter, limit=1, score_threshold=0.50,
    )
    results = response.points

    # BƯỚC 2 (FALLBACK): Nếu tìm không ra -> Vứt bỏ tiêu chí Da đi, chỉ giữ tiêu chí Dáng
    if not results and profile and profile.get("tone_da") and profile.get("dang_nguoi"):
        fallback_conditions = [
            FieldCondition(key="dang_nguoi", match=MatchAny(any=[profile["dang_nguoi"], "Mọi vóc dáng"]))
        ]
        search_filter = Filter(must=fallback_conditions)
        response = client.query_points(
            collection_name=collection, query=query_vector,
            query_filter=search_filter, limit=1, score_threshold=0.50,
        )
        results = response.points

    # BƯỚC 3 (FALLBACK): Nếu khách kén quá vẫn không có -> Vứt sạch bộ lọc đi để vớt vát 1 bộ gần giống nhất
    if not results and search_filter:
        response = client.query_points(
            collection_name=collection, query=query_vector,
            limit=1, score_threshold=0.50,
        )
        results = response.points

    # Trả về cuốn sổ cẩm nang tìm được (Hoặc None nếu cạn kiệt)
    return results[0].payload if results else None


def find_outfit_details(base_rule: dict, gender: str = "female") -> dict:
    """
    HÀM 2: TÌM CHI TIẾT TỪNG MÓN ĐỒ CHO ĐỦ BỘ
    - Mục đích: Từ công thức ở Hàm 1 (VD: Áo, Quần), đi tìm đích danh cái áo đó kiểu gì, quần kiểu gì.
    """
    collection   = f"layer_b_{gender}"
    outfit_rules = {}
    # Tạo chuỗi phong cách (VD: "Thanh lịch Đi tiệc")
    style_query  = f"{base_rule['phong_cach']} {base_rule['boi_canh']}"

    # Lặp qua từng món đồ cần mua (VD: Lặp 3 lần để tìm Áo, Quần, Giày)
    for category in base_rule.get("goi_y_phoi_cung", []):

        # CÁCH 1: Tìm Khớp Chính Xác Bằng Chữ (Cực nhanh - Không cần Qdrant)
        # Chạy ra đúng kệ đồ, tìm mác áo có dán đúng chữ "Thanh lịch" và "Đi tiệc"
        exact = [r for r in (layer_b_female if gender == "female" else layer_b_male)
                 if r["rule_key"].startswith(category)
                 and r["phong_cach"] == base_rule["phong_cach"]
                 and r["boi_canh"]   == base_rule["boi_canh"]]

        # Có thì nhặt bỏ vào giỏ ngay và sang món tiếp theo
        if exact:
            outfit_rules[category] = exact[0]
            continue

        # CÁCH 2: Dùng Qdrant tìm theo cảm quan (Semantic Search)
        # Nếu mất mác áo, phải dùng cảm quan AI để tìm đồ trông "mang âm hưởng thanh lịch đi tiệc"
        query_vector = custom_embeddings.embed_query(f"{category} {style_query}")
        
        response = client.query_points(
            collection_name = collection,
            query           = query_vector,
            query_filter    = Filter(must=[
                FieldCondition(
                    key   = "rule_key",
                    match = MatchAny(any=[f"{category} | {t}" for t in [""]])
                )
            ]),
            limit           = 1,
            score_threshold = 0.45,
        )
        results = response.points

        # Lấy hẳn 10 cái giống giống ra
        raw_response = client.query_points(
            collection_name = collection, query=query_vector,
            limit = 10, score_threshold = 0.40,
        )
        
        # Nhìn bằng mắt thường (code Python) để gạt bỏ mấy đồ không phải là "Áo", chỉ giữ lại đúng đồ cần tìm
        category_results = [r for r in raw_response.points
                            if r.payload["rule_key"].startswith(category)]
        
        # Bỏ vào giỏ đồ
        if category_results:
            outfit_rules[category] = category_results[0].payload

    # Trả về nguyên cái giỏ đồ chi tiết (Dictionary)
    return outfit_rules

print("[OK] Layer B functions đã sẵn sàng!")


[OK] Layer B functions đã sẵn sàng!


## PHẦN 6: Category Mapping — Layer B → Layer A

In [12]:
# TỪ ĐIỂN CƠ BẢN: Dịch ngôn ngữ của Stylist sang tên Kệ Hàng của Thủ kho
# Ví dụ: Stylist gọi là "Áo mặc trong" -> Thủ kho gọi là kệ "Áo"
CATEGORY_MAPPING = {
    "Áo mặc trong (áo thun/sơ mi)": ["Áo"],
    "Áo khoác ngoài":               ["Áo khoác"],
    "Áo khoác nhẹ/Áo len":          ["Áo khoác"],
    "Quần/Chân váy":                 ["Quần", "Chân váy"],
    "Đầm/Jumpsuit":                  ["Đầm", "Jumpsuit"],
    "Giày dép":                      ["Giày"],
    "Túi xách":                      ["Túi xách"],
    "Phụ kiện":                      None,  # Chữ Phụ kiện này chung chung quá, xuống từ điển dưới dịch tiếp
}

# TỪ ĐIỂN NÂNG CAO: Soi từ khóa tiếng Anh để biết Phụ Kiện đó cụ thể nằm ở kệ nào
PHU_KIEN_KEYWORD_ROUTER = {
    "Mũ":             ["beret","hat","cap","beanie","fedora","bucket","brim", "flat cap","earflap","snood","balaclava","trapper"],
    "Găng tay":        ["gloves","glove","arm warmer"],
    "Kính mắt":        ["glasses","sunglasses","sunglass"],
    "Đồng hồ":         ["watch"],
    "Dây chuyền":      ["necklace","chain pendant","chain"],
    "Bông tai":        ["earring","earrings"],
    "Vòng tay":        ["bracelet"],
    "Nhẫn":            ["ring"],
    "Ghim cài áo":     ["brooch","pin","badge"],
    "Phụ kiện hỗ trợ": ["socks","sock","scarf","tie","belt","bandana", "headband","suspender"],
}

def get_layer_a_categories(layer_b_category: str, product_type: str) -> list:
    """HÀM PHIÊN DỊCH: Đọc yêu cầu của Stylist và chỉ ra chính xác tên Kệ Hàng"""
    # Nếu không phải Phụ kiện -> Dùng ngay Từ điển Cơ Bản
    if layer_b_category != "Phụ kiện":
        return CATEGORY_MAPPING.get(layer_b_category, [])
    
    # Nếu là Phụ kiện -> Phải soi thêm từ tiếng Anh bằng Từ điển Nâng Cao
    ptype_lower = product_type.lower()
    for layer_a_cat, keywords in PHU_KIEN_KEYWORD_ROUTER.items():
        if any(kw in ptype_lower for kw in keywords):
            return [layer_a_cat] # VD: Có chữ "watch" -> Trả về kệ "Đồng hồ"
            
    # Tra không ra thì vứt đại vào kệ Phụ kiện hỗ trợ
    return ["Phụ kiện hỗ trợ"]


def get_products_for_outfit(product_type: str, layer_b_category: str,
                            phong_cach: str, vdb) -> list:
    """HÀM ĐI NHẶT HÀNG: Có thiết lập Ngưỡng (Threshold) để lọc đồ rác."""
    
    # BƯỚC 1: Phiên dịch tên kệ hàng
    target_categories = get_layer_a_categories(layer_b_category, product_type)
    
    search_filter = None
    if target_categories:
        search_filter = Filter(must=[
            FieldCondition(
                key="metadata.category",
                match=MatchAny(any=target_categories)
            )
        ])
        
    query = f"{product_type} {phong_cach}"
    
    # BƯỚC 2: Tìm kiếm nhưng lần này LẤY KÈM ĐIỂM SỐ CHẤM ĐỘ GIỐNG NHAU
    raw_results = vdb.similarity_search_with_score(query=query, k=3, filter=search_filter)
    
    # BƯỚC 3: Dò điểm. Chỉ lấy đồ nếu điểm giống nhau >= 0.35 (Ngưỡng an toàn của BGE-M3)
    valid_products = []
    for doc, score in raw_results:
        if score >= 0.35:  
            valid_products.append(doc)
            
    return valid_products



## PHẦN 7: Xử lý ảnh với Qwen2.5-VL
> Yêu cầu: `ollama pull qwen2.5vl:3b` trước khi chạy phần này.

In [13]:
QWEN_VL_MODEL = "qwen2.5vl:3b"
VL_MAX_SIZE   = 512   # px — giảm kích thước ảnh để tránh lỗi GGML_ASSERT

# Giá trị dang_nguoi + tone_da khớp với Layer B
LAYER_B_DANG_NGUOI = [
    "Dáng quả lê", "Dáng quả táo", "Dáng đồng hồ cát",
    "Dáng chữ nhật", "Dáng cân đối",
    "Người thấp bé", "Người ngoại cỡ", "Người mảnh",
]
LAYER_B_TONE_DA = [
    "Da sáng", "Da trung bình", "Da ngăm", "Da ấm",
]

LAYER_B_WILDCARD_DANG = "Mọi vóc dáng"
LAYER_B_WILDCARD_TONE = "Mọi tone da"
# ── 1. Tiền xử lý ảnh ────────────────────────────────────────────
def _preprocess_image(image_path: str) -> str:
    """
    Resize ảnh về VL_MAX_SIZE (giữ aspect ratio) và convert sang JPEG.
    Giải quyết lỗi: GGML_ASSERT(a->ne[2] * 4 == b->ne[0]) failed
    Lý do: Vision encoder của Qwen-VL có giới hạn số patch token.
           Ảnh lớn (> 1024px) vượt giới hạn → GGML crash.
    """
    img = Image.open(image_path).convert("RGB")
    w, h = img.size
    if max(w, h) > VL_MAX_SIZE:
        ratio   = VL_MAX_SIZE / max(w, h)
        img     = img.resize((int(w * ratio), int(h * ratio)), Image.LANCZOS)
    buf = io.BytesIO()
    img.save(buf, format="JPEG", quality=85)
    return base64.b64encode(buf.getvalue()).decode()


# ── 2. Gọi VL model ──────────────────────────────────────────────
def _call_vl(image_path: str, prompt: str) -> str:
    """
    Gọi Qwen2.5-VL qua Ollama.
    Ảnh được resize + convert JPEG trước để tránh lỗi GGML.
    """
    if not os.path.exists(image_path):
        raise FileNotFoundError(f"Không tìm thấy ảnh: {image_path}")
    img_b64 = _preprocess_image(image_path)
    try:
        resp = ollama.chat(
            model    = QWEN_VL_MODEL,
            messages = [{"role": "user", "content": prompt, "images": [img_b64]}],
        )
        return resp["message"]["content"].strip()
    except Exception as e:
        print(f"   ⚠️  [LỖI VISION] {e}")
        return ""


# ── 3. Phân loại ảnh ─────────────────────────────────────────────
def detect_image_type(image_path: str, user_query: str = "") -> str:
    """
    Xác định mục đích gửi ảnh dựa vào ảnh + câu hỏi đi kèm.
    Ưu tiên PRODUCT nếu user hỏi về sản phẩm (dù ảnh có người mẫu).
    """
    prompt = f"""Ảnh này chứa người hay chứa sản phẩm?
Câu hỏi của người dùng: "{user_query}"

Dựa vào câu hỏi và ảnh, xác định MỤC ĐÍCH của người dùng:
1. Muốn tìm/hỏi về quần áo/món đồ trong ảnh (dù có người mẫu mặc) → PRODUCT
2. Muốn phân tích vóc dáng/tone da của người trong ảnh             → PERSON
3. Không có câu hỏi + ảnh chủ yếu là người                        → PERSON
4. Không có câu hỏi + ảnh chụp sát sản phẩm                       → PRODUCT

Chỉ trả lời đúng 1 chữ: PERSON hoặc PRODUCT."""
    result = _call_vl(image_path, prompt).upper()
    return "product" if "PRODUCT" in result else "person"


# ── 4. Phân tích người ───────────────────────────────────────────
def analyze_person_image(image_path: str) -> dict:
    """
    Phân tích dáng người + tone da.
    Prompt dùng đúng các giá trị có trong Layer B để filter khớp.
    Parse linh hoạt: bắt cả "DÁNG:" lẫn "DÁNG NGƯỜI:" đều được.
    """
    dang_list = " | ".join(LAYER_B_DANG_NGUOI)
    tone_list = " | ".join(LAYER_B_TONE_DA)

    prompt = f"""Bạn là chuyên gia tư vấn thời trang. Phân tích người trong ảnh:

1. DÁNG NGƯỜI (chọn đúng 1 trong danh sách): {dang_list}
2. TONE DA (chọn đúng 1 trong danh sách): {tone_list}
3. NHẬN XÉT: 1-2 câu về điểm nổi bật có thể khai thác khi phối đồ.

Trả lời theo đúng format (không thêm gì khác):
DÁNG: [tên dáng]
TONE: [tên tone]
NHẬN XÉT: [nội dung]"""

    raw     = _call_vl(image_path, prompt)
    profile = {"dang_nguoi": None, "tone_da": None, "nhan_xet": ""}

    for line in raw.splitlines():
        line = line.strip()
        upper = line.upper()
        if "DÁNG" in upper and ":" in line:
            val = line.split(":", 1)[1].strip()
            profile["dang_nguoi"] = val if val else None
        elif "TONE" in upper and ":" in line:
            val = line.split(":", 1)[1].strip()
            profile["tone_da"] = val if val else None
        elif "NHẬN XÉT" in upper and ":" in line:
            profile["nhan_xet"] = line.split(":", 1)[1].strip()

    return profile


# ── 5. Caption sản phẩm ──────────────────────────────────────────
def caption_product_image(image_path: str, user_query: str = "") -> str:
    """
    Mô tả món đồ thời trang trong ảnh bằng tiếng Việt.
    user_query giúp VL tập trung đúng vào món đồ user đang hỏi.
    """
    prompt = f"""Câu hỏi của người dùng: "{user_query}"

Mô tả chi tiết MÓN ĐỒ THỜI TRANG mà người dùng đang quan tâm trong ảnh bằng tiếng Việt.
Bao gồm: loại sản phẩm, màu sắc, kiểu dáng, chất liệu (nếu nhận ra), phong cách.
Ngắn gọn 1-2 câu. TUYỆT ĐỐI KHÔNG mô tả người mẫu."""
    return _call_vl(image_path, prompt)


print(f"[OK] Vision functions đã sẵn sàng!")
print(f"     Model   : {QWEN_VL_MODEL}")
print(f"     Max size: {VL_MAX_SIZE}px  (auto resize → fix GGML_ASSERT)")
print(f"     Dáng    : {len(LAYER_B_DANG_NGUOI)} giá trị khớp Layer B")
print(f"     Tone    : {len(LAYER_B_TONE_DA)} giá trị khớp Layer B")


[OK] Vision functions đã sẵn sàng!
     Model   : qwen2.5vl:3b
     Max size: 512px  (auto resize → fix GGML_ASSERT)
     Dáng    : 8 giá trị khớp Layer B
     Tone    : 4 giá trị khớp Layer B


## PHẦN 8: Khởi tạo LLM

In [ ]:
print("[THÔNG BÁO] Đang khởi tạo LLM Qwen local qua Ollama...")

llm = ChatOllama(
    model      = "qwen3:4b-instruct",   # hoặc "qwen2.5:3b-instruct"
    temperature= 0.4,
    timeout    = 120,
    num_predict= 1024,
    num_ctx    = 8192,
)

print("[OK] LLM đã sẵn sàng!")

[THÔNG BÁO] Đang khởi tạo LLM Qwen local qua Ollama...
[OK] LLM đã sẵn sàng!


## PHẦN 9: Cấu hình Prompt & Redis

In [ ]:
# ── Prompt chính cho luồng SEARCH (anti-hallucination + xAI) ────────
system_prompt = """Bạn là một chuyên viên tư vấn thời trang cao cấp, có gu thẩm mỹ tinh tế và giọng văn vô cùng thân thiện, thanh lịch.

QUY TẮC TỐI CAO (CHỐNG BỊA ĐẶT - ANTI-HALLUCINATION):
1. BẠN PHẢI TÌM TRONG phần "Dữ LIỆU SẢN PHẨM" bên dưới để trả lời khách.
2. TUYỆT ĐỐI KHÔNG bỏa ra tên, giá tiền, hay đặc điểm sản phẩm nếu không có trong dữ liệu.
3. NẼU KHÔNG CÓ Dữ LIỆU KHẮP: Hãy xin lỗi thật duyên dáng là shop tạm hết mẫu này và chủ động hỏi khách có muốn đổi sang phong cách khác không.

CÁCH TRÌNH BÀY (Mượt mà, tự nhiên, có xAI):
- Mở đầu bằng một câu chào hoặc nhận xét nhẹ nhàng về gu của khách.
- Khi giới thiệu sản phẩm, lồng ghép thông tin khéo léo thành đoạn văn thay vì gạch đầu dòng khô khan.
- Bắt buộc in đậm **Tên Sản Phẩm** và kèm (Mã SP: [MÃ_SP]) - [Giá] VNĐ.
- xAI (GIẢI THÍCH LÝ DO - BẮT BUỘC): Sau mỗi sản phẩm, THÊM 1 câu giải thích ngắn tại sao sản phẩm này phù hợp với yêu cầu của khách (dựa vào màu sắc, chất liệu, dịp mặc, hoặc vóc dáng).
- Nếu ĐẦU ẢNH có trong dữ liệu: đính kèm theo format ![Sản phẩm](URL_ẢNH) để khách xem trực quan.
- Trả lời súc tích, không vượt quá 300 từ.

Dữ LIỆU SẢN PHẨM:
{context}"""

QA_PROMPT = ChatPromptTemplate.from_messages([
    ("system", system_prompt),
    MessagesPlaceholder("chat_history"),
    ("human", "{input}"),
])

# ── Prompt viết lại câu hỏi (query rewriting) ─────────────────
contextualize_q_prompt = ChatPromptTemplate.from_messages([
    ("system", """Nhiệm vụ của bạn là NGƯỜI VIẾT LẠI CÂU Hỏi. 
Dựa vào lịch sử trò chuyện, hãy làm rõ nghĩa của câu hỏi mới nhất để nó có thể đứng độc lập mà ai đọc cũng hiểu được.

QUY TẮC SỐNG CÒN:
- TUYỆT ĐỐI KHÔNG TRẢ LỜI CÂU Hỏi CỦA KHÁCH.
- CHỈ IN RA DUY NHẤT CÂU Hỏi ĐÃ ĐƯỢC VIẾT LẠI. Không giải thích, không dạ thưa, cấm thêm chữ \"Câu hỏi là:\".
- Nếu câu hỏi đã quá rõ ràng rồi, hãy in lại y nguyên câu đó.

VÍ DỤ: 
Khách: \"Có màu khác không?\" -> BẠN CHỈ ĐƯỢC IN RA: \"Ao thun đỏ ở trên có màu khác không?\"
"""),
    MessagesPlaceholder("chat_history"),
    ("human", "{input}"),
])

# ── Document format gửi vào LLM (đã có ảnh) ───────────────────
doc_prompt = PromptTemplate.from_template(
    "\n[MÃ_SP: {product_id}]"
    "\nẢNH: {images}"
    "\nTHÔNG TIN CHI TIẾT: {page_content}\n"
)

# ── Prompt cho luồng OUTFIT (Personal Stylist + xAI) ───────────
OUTFIT_SYSTEM_PROMPT = """Bạn là một chuyên gia tạo dáng (Personal Stylist) cực kỳ chuyên nghiệp và tâm lý.

NHIỆM VỤ: Dựa vào \"CÔNG THỨC PHỐI ĐỒ\" và \"SẢN PHẨM GỢI Ý\" bên dưới, hãy \"hô biến\" một bộ outfit hoàn hảo cho khách hàng.

QUY TẮC:
1. Khéo léo xâu chuỗi các món đồ thành một bức tranh tổng thể.
2. TUYỆT ĐỐI không giới thiệu đồ ngoài danh sách \"SẢN PHẨM GỢI Ý\". Không tự bỏ thêm đồ.
3. Nhớ in đậm **Tên Sản Phẩm**, kèm (Mã SP: [MÃ_SP]) và [Giá] VNĐ ở mỗi món.
4. xAI - TÍNH MINH BẠCH (BẮT BUỘC): ở mỗi món đồ, BẠN PHẢI GIẢI THÍCH TẠI SAO không đoly này phù hợp (dựa vào vóc dáng, tone da, hoặc lý do có trong công thức). 
5. Nếu ĐẦU ẢNH có trong dữ liệu sản phẩm: đính kèm ![Sản phẩm](URL_ẢNH) để khách xem trực quan.
6. Giọng điệu niịnh khách, sang trọng nhưng gần gũi. Lồng ghép thành các đoạn văn mượt mà, tránh dùng gạch đầu dòng liệt kê như hóa đơn.
7. Kết thúc bằng 1 câu chốt sale/hỏi han thân thiện. Giới hạn 350 từ.

{outfit_context}"""

outfit_prompt = ChatPromptTemplate.from_messages([
    ("system", OUTFIT_SYSTEM_PROMPT),
    MessagesPlaceholder("chat_history"),
    ("human", "{input}"),
])

# ── Redis lưu lịch sử hội thoại ─────────────────────────────
REDIS_URL = "redis://localhost:6379"

# def get_message_history(session_id: str):
#     history = RedisChatMessageHistory(session_id, url=REDIS_URL)
#     current_messages = history.messages
#     if len(current_messages) > 6:           # giữ 6 message gần nhất (3 lượt chat)
#         kept = current_messages[-6:]
#         history.clear()
#         history.add_messages(kept)
#     return history
# Cell 21 — thay get_message_history() bằng version có summarization

SUMMARIZE_PROMPT = """Tóm tắt cuộc hội thoại mua sắm thời trang sau thành 3-5 câu ngắn.
Giữ lại: sản phẩm đã hỏi, phong cách khách thích, thông tin vóc dáng/tone da (nếu có).
Bỏ qua: lời chào, câu xã giao.
Chỉ trả về đoạn tóm tắt, không thêm gì khác.

Hội thoại:
{history_text}"""

def summarize_history(messages: list) -> str:
    """Dùng LLM tóm tắt lịch sử hội thoại cũ."""
    history_text = "\n".join([
        f"{'Khách' if m.type == 'human' else 'Bot'}: {m.content[:300]}"
        for m in messages
    ])
    resp = ollama.chat(
        model   = "qwen3:4b-instruct",
        messages= [{"role": "user",
                    "content": SUMMARIZE_PROMPT.format(history_text=history_text)}],
        options = {"temperature": 0, "num_predict": 150}
    )
    return resp["message"]["content"].strip()


def get_message_history(session_id: str):
    from langchain_core.messages import SystemMessage

    history  = RedisChatMessageHistory(session_id, url=REDIS_URL)
    messages = history.messages

    # Dưới 8 message → giữ nguyên, chưa cần tóm tắt
    if len(messages) <= 8:
        return history

    # Trên 8 message → tóm tắt phần cũ, giữ 4 message gần nhất
    old_messages    = messages[:-4]
    recent_messages = messages[-4:]

    summary_text = summarize_history(old_messages)

    # Rebuild history: [summary message] + [4 recent messages]
    history.clear()
    history.add_message(SystemMessage(
        content=f"[TÓM TẮT HỘI THOẠI TRƯỚC]: {summary_text}"
    ))
    history.add_messages(recent_messages)

    return history
print("[OK] Prompts & Redis đã nâng cấp: xAI + ảnh inline + giọng văn thân thiện!")


[OK] Prompts & Redis đã nâng cấp lên phiên bản Sang Trọng & Tự Nhiên!


## PHẦN 10: Lắp ráp RAG Pipeline

In [16]:
print("[THÔNG BÁO] Đang lắp ráp RAG Pipeline...")

# ── Chain 1: Luồng SEARCH (RAG đầy đủ) ────────────────────────────
history_aware_retriever = create_history_aware_retriever(
    llm, retriever, contextualize_q_prompt
)
document_chain = create_stuff_documents_chain(
    llm=llm, prompt=QA_PROMPT, document_prompt=doc_prompt
)
rag_chain = create_retrieval_chain(history_aware_retriever, document_chain)

full_chat_chain = RunnableWithMessageHistory(
    rag_chain, get_message_history,
    input_messages_key="input",
    history_messages_key="chat_history",
    output_messages_key="answer",
)

# ── Chain 2: Luồng OUTFIT (không dùng retriever — context đã build sẵn) ──
outfit_llm_chain = outfit_prompt | llm
outfit_chain_with_history = RunnableWithMessageHistory(
    outfit_llm_chain, get_message_history,
    input_messages_key="input",
    history_messages_key="chat_history",
)

print("[OK] RAG Pipeline đã sẵn sàng!")

[THÔNG BÁO] Đang lắp ráp RAG Pipeline...
[OK] RAG Pipeline đã sẵn sàng!


## PHẦN 11: Intent Detection

In [17]:
# ═══════════════════════════════════════════════════════════════════
# PHẦN 11: INTENT DETECTION — Hybrid (Keyword + LLM)
# ═══════════════════════════════════════════════════════════════════
#
# Chiến lược 2 tầng:
#   Tầng 1 — Keyword "chắc chắn": xử lý ngay, không gọi LLM → nhanh
#   Tầng 2 — LLM classify: chỉ gọi khi câu mơ hồ, có thêm context
#             của bot message trước để hiểu đúng hơn
# ═══════════════════════════════════════════════════════════════════

# ── Tầng 1: Keyword list "chắc chắn" ─────────────────────────────
# Chỉ đưa vào đây những từ KHÔNG BAO GIỜ bị nhầm intent

DEFINITE_GREETING = [
    "xin chào", "hello", "hi bạn", "chào bạn", "hey", "alo",
    "chào buổi sáng", "chào buổi chiều",
]
DEFINITE_CHITCHAT = [
    "cảm ơn", "cảm on", "thank you", "thanks", "tạm biệt",
    "bye", "hẹn gặp lại", "bái bai",
]
DEFINITE_OUTFIT = [
    "phối đồ", "mix match", "mặc với gì", "mặc cùng gì",
    "kết hợp với gì", "phối với gì", "outfit cho",
    "gợi ý outfit", "tư vấn phối",
]
DEFINITE_SEARCH = [
    "còn hàng không", "còn size", "giá bao nhiêu", "mã sp",
    "có bán không", "tìm giúp", "cho xem", "shop có",
]

MALE_KEYWORDS = ["nam", "con trai", "anh", "bạn trai", "chàng", "đàn ông"]

# ── Tầng 2: LLM classify ─────────────────────────────────────────
INTENT_CLASSIFY_PROMPT = """Bạn là bộ phân loại intent cho chatbot tư vấn thời trang.
Phân loại câu hỏi vào đúng 1 trong 4 nhóm:

OUTFIT   → Hỏi cách phối đồ, mix-match, tư vấn mặc gì cho dịp/vóc dáng/phong cách
SEARCH   → Tìm sản phẩm cụ thể, hỏi giá, còn hàng, so sánh, xem ảnh sản phẩm
CHITCHAT → Cảm ơn, tạm biệt, hỏi thăm, câu xã giao không liên quan mua sắm
GREETING → Chào hỏi, bắt đầu cuộc trò chuyện

Ví dụ bắt buộc phải học thuộc:
"có áo nào mặc đi tiệc không?"          → SEARCH   (tìm sản phẩm, dù có từ "mặc")
"tìm đồ phù hợp với quần jean xanh"     → OUTFIT   (phối đồ, không phải tìm 1 sản phẩm đơn)
"tôi cao 1m55 nên mặc gì?"              → OUTFIT   (tư vấn theo vóc dáng)
"muốn trông thanh lịch hơn thì sao?"    → OUTFIT   (tư vấn phong cách)
"áo này với quần kia trông có hợp không?"→ OUTFIT   (hỏi về sự phối hợp)
"còn size L không?"                     → SEARCH   (hỏi tồn kho)
"oke cảm ơn bạn nhiều"                  → CHITCHAT
"đi làm"                                → OUTFIT   (trả lời câu hỏi "dịp nào" của bot)
{context_block}
Câu cần phân loại: "{query}"

Chỉ trả lời đúng 1 từ: OUTFIT / SEARCH / CHITCHAT / GREETING"""


def detect_intent_llm(query: str, last_bot_msg: str = "") -> str:
    """
    Dùng LLM phân loại intent cho những câu mơ hồ mà keyword không xử lý được.
    Truyền thêm last_bot_msg để LLM hiểu context (VD: user trả lời câu hỏi của bot).
    """
    context_block = ""
    if last_bot_msg:
        context_block = (
            f'\nContext — Bot vừa nói: "{last_bot_msg[:120]}..."\n'
        )

    try:
        resp = ollama.chat(
            model   = "qwen3:4b-instruct",
            messages= [{
                "role"   : "user",
                "content": INTENT_CLASSIFY_PROMPT.format(
                    query         = query,
                    context_block = context_block,
                )
            }],
            options = {
                "temperature": 0,        # deterministic — không cần sáng tạo
                "num_predict": 10,       # chỉ cần 1 từ → rất nhanh (~0.3-0.5s)
            }
        )
        result = resp["message"]["content"].strip().upper()

        # Parse kết quả — lấy intent đầu tiên xuất hiện trong response
        for intent in ["OUTFIT", "SEARCH", "CHITCHAT", "GREETING"]:
            if intent in result:
                return intent.lower()

    except Exception as e:
        print(f"[WARN] LLM intent classify lỗi: {e} → fallback search")

    return "search"   # fallback an toàn


def detect_intent(query: str, last_bot_msg: str = "") -> str:
    """
    Hybrid intent detection:
      1. Keyword "chắc chắn" → trả về ngay (không gọi LLM, không tốn thời gian)
      2. Câu mơ hồ          → gọi LLM classify với context bot message trước
    """
    q = query.lower().strip()

    # Tầng 1: keyword chắc chắn (theo thứ tự ưu tiên)
    # if any(kw in q for kw in DEFINITE_GREETING):  return "greeting"
    # if any(kw in q for kw in DEFINITE_CHITCHAT):  return "chitchat"
    # if any(kw in q for kw in DEFINITE_OUTFIT):    return "outfit"
    # if any(kw in q for kw in DEFINITE_SEARCH):    return "search"
    if any(kw in q for kw in DEFINITE_OUTFIT):    return "outfit"
    if any(kw in q for kw in DEFINITE_SEARCH):    return "search"
    if any(kw in q for kw in DEFINITE_GREETING):  return "greeting"
    if any(kw in q for kw in DEFINITE_CHITCHAT):  return "chitchat"
    # Tầng 2: câu mơ hồ → LLM
    return detect_intent_llm(query, last_bot_msg)


def detect_gender(query: str) -> str:
    """Trả về 'male' nếu query đề cập giới tính nam, mặc định 'female'."""
    return "male" if any(kw in query.lower() for kw in MALE_KEYWORDS) else "female"


def get_greeting_response() -> str:
    return (
        "Xin chào! Mình là trợ lý tư vấn thời trang của shop. "
        "Bạn cần tìm sản phẩm hay muốn được gợi ý phối đồ hôm nay? 😊"
    )

def get_chitchat_response(query: str) -> str:
    return "Rất vui được hỗ trợ bạn! Bạn còn muốn hỏi thêm gì về thời trang không?"


# ── Test nhanh ────────────────────────────────────────────────────
print("[OK] Intent Detection (Hybrid) đã sẵn sàng!")
print("     Tầng 1 (keyword): greeting / chitchat / outfit / search")
print("     Tầng 2 (LLM)    : xử lý câu mơ hồ + context bot message")

TEST_CASES = [
    ("xin chào bạn",                         "greeting"),
    ("cảm ơn bạn nhiều nha",                 "chitchat"),
    ("phối đồ với áo thun trắng",            "outfit"),
    ("còn size L không",                     "search"),
    ("tôi cao 1m55 nên mặc gì?",             "outfit → LLM"),
    ("có áo nào mặc đi tiệc không?",         "search → LLM"),
    ("muốn trông thanh lịch hơn thì sao?",   "outfit → LLM"),
    ("đi làm",                               "outfit → LLM (cần context)"),
]

print("\n--- Kiểm tra Tầng 1 (không gọi LLM) ---")
for query, expected in TEST_CASES:
    q = query.lower().strip()
    tier1 = None
    if any(kw in q for kw in DEFINITE_GREETING): tier1 = "greeting"
    elif any(kw in q for kw in DEFINITE_CHITCHAT): tier1 = "chitchat"
    elif any(kw in q for kw in DEFINITE_OUTFIT): tier1 = "outfit"
    elif any(kw in q for kw in DEFINITE_SEARCH): tier1 = "search"

    if tier1:
        status = "✅" if tier1 == expected.split(" →")[0] else "⚠️"
        print(f"  {status} [{tier1:8s}] ← keyword | '{query}'")
    else:
        print(f"  🔁 [ambiguous] → cần LLM  | '{query}' (expect: {expected})")


[OK] Intent Detection (Hybrid) đã sẵn sàng!
     Tầng 1 (keyword): greeting / chitchat / outfit / search
     Tầng 2 (LLM)    : xử lý câu mơ hồ + context bot message

--- Kiểm tra Tầng 1 (không gọi LLM) ---
  ✅ [greeting] ← keyword | 'xin chào bạn'
  ✅ [chitchat] ← keyword | 'cảm ơn bạn nhiều nha'
  ✅ [outfit  ] ← keyword | 'phối đồ với áo thun trắng'
  ✅ [search  ] ← keyword | 'còn size L không'
  🔁 [ambiguous] → cần LLM  | 'tôi cao 1m55 nên mặc gì?' (expect: outfit → LLM)
  🔁 [ambiguous] → cần LLM  | 'có áo nào mặc đi tiệc không?' (expect: search → LLM)
  🔁 [ambiguous] → cần LLM  | 'muốn trông thanh lịch hơn thì sao?' (expect: outfit → LLM)
  🔁 [ambiguous] → cần LLM  | 'đi làm' (expect: outfit → LLM (cần context))


## PHẦN 12: Xây dựng Outfit Context (Layer B → Layer A)

In [18]:
def build_outfit_context(user_query: str,
                         gender: str   = "female",
                         profile: dict = None) -> str:
    """
    TỔNG QUẢN LÝ: Xâu chuỗi Layer B và Layer A thành Hồ sơ hoàn chỉnh.
    """
    # Bước 1: Layer B lần 1
    base_rule = find_matching_rule(user_query, gender, profile)
    if not base_rule:
        return ""

    # Bước 2: Layer B lần 2
    outfit_rules = find_outfit_details(base_rule, gender)
    if not outfit_rules:
        return ""

    # Bước 3: Layer A
    outfit_products = {}
    for layer_b_category, rule in outfit_rules.items():
        product_type = rule["rule_key"].split("|")[1].strip()
        products     = get_products_for_outfit(
            product_type, layer_b_category, base_rule["phong_cach"], vector_db
        )
        outfit_products[layer_b_category] = {
            "product_type": product_type,
            "ly_do":        rule["ly_do_tu_van"],
            "products":     products,
        }

    # Bước 4: Build context string
    lines = [
        "CÔNG THỨC PHỐI ĐỒ:",
        f"  Phong cách : {base_rule['phong_cach']}",
        f"  Bối cảnh   : {base_rule['boi_canh']}",
        f"  Lý do chính: {base_rule['ly_do_tu_van']}",
    ]
    if profile and profile.get("dang_nguoi"):
        lines.append(f"  Dáng người : {profile['dang_nguoi']}")
    if profile and profile.get("tone_da"):
        lines.append(f"  Tone da    : {profile['tone_da']}")
    lines += ["", "SẢN PHẨM GỢI Ý:"]

    for cat, data in outfit_products.items():
        lines.append(f"\n[{cat} – {data['product_type']}]")
        lines.append(f"  Lý do: {data['ly_do']}")
        if data["products"]:
            for doc in data["products"]:
                pid   = doc.metadata.get("product_id", "N/A")
                price_raw = doc.metadata.get("price", "N/A")
                
                # CẢI TIẾN 1: Format giá tiền (Ví dụ: 250000 -> 250.000)
                try:
                    price_fmt = f"{int(price_raw):,}".replace(",", ".")
                except:
                    price_fmt = price_raw
                
                lines.append(f"  • (Mã SP: {pid} | Giá: {price_fmt} VNĐ)")
                
                # CẢI TIẾN 2: Thả rông mô tả sản phẩm (Lấy 600 chữ cái thay vì 200 để AI đủ dữ kiện tư vấn)
                lines.append(f"    {doc.page_content[:600]}")
        else:
            lines.append("  • (Chưa có sản phẩm phù hợp trong kho)")

    return "\n".join(lines)

print("[OK] Tổng quản lý build_outfit_context đã sẵn sàng!")


[OK] Tổng quản lý build_outfit_context đã sẵn sàng!


## PHẦN 13: Luồng chạy chính (Chat Loop)

In [19]:
SESSION_ID   = str(uuid.uuid4())
user_profile = {}  # Tích lũy dáng người + tone da qua từng lượt chat

class SpyRetrieverHandler(BaseCallbackHandler):
    """Hiển thị câu hỏi sau khi LLM đã viết lại (debug)."""
    def on_retriever_start(self, serialized: dict, query: str, **kwargs):
        print(f"\n🕵️  [Câu hỏi sau rewrite]: {query}\n")

print("=" * 60)
print("  👗👔  CHATBOT TƯ VẤN THỜI TRANG  👔👗  ")
print("     Nhập '0' để thoát | Nhập đường dẫn ảnh nếu có")
print("=" * 60 + "\n")

while True:
    # ── Nhận input ──────────────────────────────────────────────────
    user_input = input("👤 Bạn: ").strip()
    if user_input == "0":
        print("\n🤖 Chatbot: Cảm ơn bạn đã ghé thăm shop. Hẹn gặp lại! 👋")
        break
    if not user_input:
        continue

    final_query = user_input

    # ── Xử lý ảnh (nếu có) ──────────────────────────────────────────
    raw_img = input("📎 Ảnh (Enter để bỏ qua): ").strip()
    if raw_img:
        if not os.path.exists(raw_img):
            print(f"   ⚠️  Không tìm thấy file: {raw_img}")
        else:
            print("🔍 [Đang phân tích ảnh...]")
            image_type = detect_image_type(raw_img, user_input)
            print(f"   → Phát hiện: {image_type.upper()}")

            if image_type == "person":
                # Phân tích vóc dáng → lưu profile → trả lời ngay
                person_info = analyze_person_image(raw_img)
                if person_info["dang_nguoi"]:
                    user_profile["dang_nguoi"] = person_info["dang_nguoi"]
                if person_info["tone_da"]:
                    user_profile["tone_da"] = person_info["tone_da"]

                print(f"\n🤖 Chatbot: Mình đã phân tích xong! "
                      f"Bạn có **{person_info['dang_nguoi']}** "
                      f"với **{person_info['tone_da']}**. "
                      f"{person_info['nhan_xet']} "
                      f"\n\nMình đã lưu thông tin này để tư vấn phối đồ phù hợp hơn cho bạn. "
                      f"Bạn muốn gợi ý outfit cho dịp nào?")
                print("\n" + "-" * 60 + "\n")
                continue  # Không chạy pipeline RAG lượt này

            else:
                # Ảnh sản phẩm → caption → làm query
                caption = caption_product_image(raw_img, user_input)
                print(f"   → Caption: {caption[:80]}...")
                final_query = f"{caption}. Yêu cầu: {user_input}" if user_input else caption

    # ── Phát hiện intent & gender ────────────────────────────────────
    intent = detect_intent(final_query)
    
    # --- CẢI TIẾN CHỐNG MẤT TRÍ NHỚ GIỚI TÍNH ---
    current_gender = detect_gender(final_query)
    
    # Nếu câu chat có chữ "nam", lưu ngay vào sổ tay để nhớ mãi mãi
    if current_gender == "male":
        user_profile["gender"] = "male"
    
    # Lúc tìm đồ: Ưu tiên lấy giới tính trong sổ tay, nếu sổ trống thì mặc định là Nữ
    gender = user_profile.get("gender", current_gender)
    # ----------------------------------------------
    
    profile_status = f"Giới: {gender.upper()} | Dáng: {user_profile.get('dang_nguoi','?')} | Tone: {user_profile.get('tone_da','?')}" if user_profile else "chưa có"

    print(f"🔍 [Intent: {intent.upper()} | Gender: {gender} | Profile: {profile_status}]")

    # ── Greeting & Chitchat (không gọi RAG) ─────────────────────────
    if intent == "greeting":
        print(f"🤖 Chatbot: {get_greeting_response()}")
        print("\n" + "-" * 60 + "\n")
        continue

    if intent == "chitchat":
        print(f"🤖 Chatbot: {get_chitchat_response(final_query)}")
        print("\n" + "-" * 60 + "\n")
        continue

    # ── Pipeline RAG (search hoặc outfit) ───────────────────────────
    print("🤖 Chatbot: ", end="")
    start_time       = time.time()
    first_token_time = None

    try:
        if intent == "outfit":
            outfit_context = build_outfit_context(final_query, gender, user_profile)
            if not outfit_context:
                print("[Layer B không khớp → fallback RAG search]")
                intent = "search"   # fallback
            else:
                for chunk in outfit_chain_with_history.stream(
                    {"input": user_input, "outfit_context": outfit_context},
                    config={"configurable": {"session_id": SESSION_ID}},
                ):
                    token = chunk.content if hasattr(chunk, "content") else str(chunk)
                    if token:
                        if first_token_time is None:
                            first_token_time = time.time()
                        print(token, end="", flush=True)

        if intent == "search":
            for chunk in full_chat_chain.stream(
                {"input": final_query},
                config={
                    "configurable": {"session_id": SESSION_ID},
                    "callbacks":    [SpyRetrieverHandler()],
                },
            ):
                if "answer" in chunk:
                    if first_token_time is None:
                        first_token_time = time.time()
                    print(chunk["answer"], end="", flush=True)

        end_time = time.time()
        if first_token_time is None:
            first_token_time = end_time

        print(f"\n\n⏱️  TTFT: {first_token_time - start_time:.2f}s | "
              f"Total: {end_time - start_time:.2f}s")

    except Exception as e:
        print(f"\n[LỖI] {e}")

    print("\n\n" + "-" * 60 + "\n")

  👗👔  CHATBOT TƯ VẤN THỜI TRANG  👔👗  
     Nhập '0' để thoát | Nhập đường dẫn ảnh nếu có


🤖 Chatbot: Cảm ơn bạn đã ghé thăm shop. Hẹn gặp lại! 👋
